# SPPACEU — 6

Run from the repository directory after installing `requirements.txt`. Configuration uses environment variables described in README.md. External processing is disabled by default. Choose one video-screening variant (steps 3–7); do not run all variants against the same outputs.


In [ ]:
import os
from pathlib import Path
from notebook_runtime import DATA_DIR, RECORDS_DIR, MODEL_NAME, load_openai_key, prepare_run

RUN_ANALYSIS = False  # Enable explicitly after reviewing inputs and API/download settings.
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import re
import time
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple, Set

import pandas as pd
from openai import OpenAI


# =========================
# CONFIGURATION
# =========================

INPUT_XLSX = DATA_DIR / "youtube_data.xlsx"

CHANNEL_AI_XLSX = DATA_DIR / "channel_ai_analysis.xlsx"
PLAYLIST_AI_XLSX = DATA_DIR / "playlist_ai_analysis.xlsx"

OUTPUT_XLSX = DATA_DIR / "video_ai_analysis_no_shorts_step_6.xlsx"
LOG_XLSX = DATA_DIR / "video_ai_processing_log_no_shorts_step_6.xlsx"
DIAGNOSTICS_XLSX = DATA_DIR / "video_selection_diagnostics_no_shorts_step_6.xlsx"

API_KEY_PATH = None  # Read OPENAI_API_KEY from the environment.

SHEET_UPLOADS = "uploads"
SHEET_PLAYLIST_MEMBERSHIP = "playlist_video_membership"

MODEL_NAME = os.environ.get("OPENAI_MODEL", "").strip()

MAX_RETRIES = 3
REQUEST_TIMEOUT = 180
SLEEP_BETWEEN_REQUESTS = 1.0
MAX_COMPLETION_TOKENS = 3500
SKIP_EXISTING = True

LIMIT_ROWS: Optional[int] = None

# First run as dry run. Then set DRY_RUN_ONLY=False and MAX_AI_ROWS=300 for a test.
MAX_AI_ROWS: Optional[int] = 0
DRY_RUN_ONLY = True

# Date filter: full-scale war period.
DATE_FILTER_ENABLED = True
DATE_COLUMN = "published_at"
DATE_FROM = "2022-02-24"
DATE_TO = "2026-12-31"

# Short video exclusion.
EXCLUDE_SHORTS_ENABLED = True
SHORT_MAX_DURATION_SECONDS = 90

CHANNEL_RELEVANCE_THRESHOLD = 0.85
PLAYLIST_RELEVANCE_THRESHOLD = 0.85

RANDOM_CONTROL_SAMPLE_SIZE = 500
RANDOM_SEED = 42

MIN_CONTEXT_ONLY_HITS = 2

FILTER_STATS: Dict[str, Any] = {}

STRONG_DIRECT_KEYWORD_GROUPS: Set[str] = {
    "corruption_direct",
    "anti_corruption_institutions",
}

WEAK_DIRECT_KEYWORD_GROUPS: Set[str] = {
    "public_audit_control",
    "accountability_transparency",
    "public_money_procurement",
}

DIRECT_KEYWORD_GROUPS: Set[str] = {
    "corruption_direct",
    "anti_corruption_institutions",
    "public_audit_control",
    "accountability_transparency",
    "public_money_procurement",
}

CONTEXT_ONLY_KEYWORD_GROUPS: Set[str] = {
    "government_officials",
    "ukraine_context",
}


# =========================
# REQUIRED COLUMNS
# =========================

REQUIRED_UPLOADS_COLUMNS = [
    "video_id",
    "channel_id",
    "channel_title",
    "video_title",
]


# =========================
# KEYWORD DICTIONARY
# =========================

KEYWORD_GROUPS: Dict[str, List[str]] = {
    "corruption_direct": [
        "корупція", "коррупция", "corruption",
        "корупцій", "коррупцион", "корумпован",
        "хабар", "взятка", "bribe", "bribery",
        "відкат", "откат", "kickback",
        "розкрадання", "хищение", "крадіжка", "воровство",
        "розпил", "дерибан",
        "зловживання", "злоупотребление",
        "незаконне збагачення", "незаконное обогащение",
    ],
    "anti_corruption_institutions": [
        "набу", "nabu",
        "сап", "sap",
        "назк", "nazk", "нaзк",
        "дбр", "dbr",
        "вакc", "вакс", "hacc",
        "антикорупційний суд", "антикоррупционный суд",
        "антикорупцій", "антикоррупци",
    ],
    "public_audit_control": [
        "публічний аудит", "публичный аудит", "public audit",
        "аудит", "audit",
        "моніторинг", "мониторинг", "monitoring",
        "громадський контроль", "общественный контроль",
        "контроль влади", "контроль власти",
        "watchdog", "викривач", "викривачі", "разоблачитель", "whistleblower",
        "викриття", "разоблачение", "exposure",
        "розслідування", "расследование", "investigation",
        "журналістське розслідування", "журналистское расследование",
    ],
    "accountability_transparency": [
        "підзвітність", "подотчетность", "accountability",
        "прозорість", "прозрачность", "transparency",
        "декларація", "декларации", "декларування",
        "конфлікт інтересів", "конфликт интересов",
        "доброчесність", "добропорядочность", "integrity",
        "відкриті дані", "открытые данные", "open data",
    ],
    "public_money_procurement": [
        "тендер", "tender",
        "закупівлі", "закупки", "procurement",
        "prozorro", "прозорро",
        "бюджет", "державні кошти", "государственные деньги",
        "публічні кошти", "публичные средства",
        "грант", "субвенція", "субсидія",
        "контракт", "підряд", "подряд",
    ],
    "government_officials": [
        "чиновник", "посадовець", "должностное лицо",
        "депутат", "міністр", "министр",
        "мер", "голова громади", "глава громады",
        "суддя", "судья", "прокурор",
        "держслужбовець", "госслужащий",
        "влада", "власть", "уряд", "правительство",
    ],
    "ukraine_context": [
        "україна", "украина", "ukraine",
        "україн", "украин",
        "київ", "киев", "kyiv", "kiev",
        "верховна рада", "рада",
        "зеленський", "зеленский",
        "кабмін", "кабмин",
    ],
}


# =========================
# SYSTEM PROMPT
# =========================

SYSTEM_INSTRUCTION = """
You are analyzing YouTube video metadata for a sociological research project.

Research topic:
"Social practices of public audit and corruption exposure in Ukraine."

Your task is to classify one YouTube video based only on the provided metadata:
- video title;
- video description, if available;
- tags and topic categories, if available;
- channel title;
- publication date, duration, format, and engagement indicators;
- optional channel-level and playlist-level AI classifications.

Do not invent facts. Do not use external knowledge about the channel, author, or actual video content.
Classify only what is supported by the metadata.

Return ONLY valid JSON. No markdown. No explanatory text outside JSON.

Use this JSON schema:

{
  "video_topic": "string",
  "secondary_topics": ["string"],
  "video_function": "string",
  "video_format_type": "string",
  "corruption_relevance_category": "string",
  "public_audit_relevance_category": "string",
  "ukraine_relevance_category": "string",
  "evidence_type": "string",
  "video_relevance_score": 0.0,
  "classification_confidence": 0.0,
  "inclusion_status": "string",
  "manual_review_required": true,
  "evidence_markers": ["string"],
  "risk_notes": ["string"],
  "short_rationale": "string"
}

video_function must be one of:
- investigative
- analytical
- news
- civic_monitoring
- public_accountability
- anti_corruption_education
- legal_or_judicial_information
- political_commentary
- civic_activism
- interview
- report
- documentary
- archive
- entertainment
- promotional
- mixed
- unclear

video_format_type must be one of:
- long_video
- short_video
- stream_or_live_recording
- podcast_episode
- news_segment
- interview
- excerpt_or_clip
- unclear

corruption_relevance_category must be one of:
- direct_corruption_focus
- indirect_governance_or_accountability_focus
- possible_contextual_relevance
- low_relevance
- unclear

public_audit_relevance_category must be one of:
- direct_public_audit_focus
- civic_monitoring_or_watchdog_focus
- transparency_or_accountability_focus
- possible_contextual_relevance
- low_relevance
- unclear

ukraine_relevance_category must be one of:
- direct_ukraine_focus
- ukrainian_language_or_context
- possible_ukraine_context
- no_visible_ukraine_focus
- unclear

evidence_type must be one of:
- direct_title_evidence
- title_and_description_evidence
- description_only_evidence
- keyword_only_evidence
- channel_or_playlist_context_only
- weak_or_ambiguous_evidence

video_relevance_score:
0.80-1.00 = highly relevant
0.60-0.79 = relevant but requires review
0.40-0.59 = partially/contextually relevant
0.00-0.39 = low relevance

short_rationale:
2-4 sentences explaining the classification in academic language.
""".strip()


# =========================
# BASIC HELPERS
# =========================

def load_api_key() -> str:
    return load_openai_key(API_KEY_PATH)


def safe_str(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def truncate_text(text: Any, max_chars: int) -> str:
    text = safe_str(text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rstrip() + "..."


def validate_columns(df: pd.DataFrame, required: List[str], sheet_name: str) -> None:
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns in sheet '{sheet_name}': {missing}")


def list_to_pipe(value: Any) -> str:
    if isinstance(value, list):
        return " | ".join(str(x) for x in value)

    return safe_str(value)


def split_groups(groups_text: str) -> Set[str]:
    if not groups_text:
        return set()

    return set(
        x.strip()
        for x in str(groups_text).split("|")
        if x.strip()
    )


def truthy_series(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y", "так"})
    )


def detect_shorts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    reasons = pd.Series("", index=df.index, dtype="object")
    is_short = pd.Series(False, index=df.index)

    if "source_section" in df.columns:
        source_short = df["source_section"].fillna("").astype(str).str.lower().str.contains("short", na=False)
        is_short = is_short | source_short
        reasons = reasons.mask(source_short, reasons + "|source_section")

    if "api_section" in df.columns:
        api_short = df["api_section"].fillna("").astype(str).str.lower().str.contains("short", na=False)
        is_short = is_short | api_short
        reasons = reasons.mask(api_short, reasons + "|api_section")

    if "is_possible_short" in df.columns:
        possible_short = truthy_series(df["is_possible_short"])
        is_short = is_short | possible_short
        reasons = reasons.mask(possible_short, reasons + "|is_possible_short")

    if "duration_seconds" in df.columns:
        duration = pd.to_numeric(df["duration_seconds"], errors="coerce")
        duration_short = duration.notna() & (duration <= SHORT_MAX_DURATION_SECONDS)
        is_short = is_short | duration_short
        reasons = reasons.mask(duration_short, reasons + "|duration_seconds<=90")

    df["_is_short_for_filter"] = is_short
    df["_short_filter_reason"] = (
        reasons
        .str.strip("|")
        .str.replace(r"\|+", "|", regex=True)
    )

    return df


# =========================
# LOAD DATA
# =========================

def load_uploads() -> pd.DataFrame:
    global FILTER_STATS

    if not INPUT_XLSX.exists():
        raise FileNotFoundError(f"Input Excel file not found: {INPUT_XLSX}")

    df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_UPLOADS)
    validate_columns(df, REQUIRED_UPLOADS_COLUMNS, SHEET_UPLOADS)

    df["video_id"] = df["video_id"].astype(str).str.strip()
    df["channel_id"] = df["channel_id"].astype(str).str.strip()

    initial_count = len(df)
    FILTER_STATS["rows_initial"] = initial_count

    if DATE_FILTER_ENABLED:
        if DATE_COLUMN not in df.columns:
            raise ValueError(f"Date filter is enabled, but column '{DATE_COLUMN}' is missing.")

        df["_published_at_dt"] = pd.to_datetime(
            df[DATE_COLUMN],
            errors="coerce",
            utc=True
        )

        date_from = pd.to_datetime(DATE_FROM, utc=True)
        date_to = pd.to_datetime(DATE_TO, utc=True)

        before_count = len(df)

        df = df[
            (df["_published_at_dt"] >= date_from)
            & (df["_published_at_dt"] <= date_to)
        ].copy()

        df["_published_at_dt_excel"] = df["_published_at_dt"].dt.tz_localize(None)

        after_count = len(df)

        FILTER_STATS["rows_before_date_filter"] = before_count
        FILTER_STATS["rows_after_date_filter"] = after_count
        FILTER_STATS["rows_removed_by_date_filter"] = before_count - after_count

        print(
            f"Date filter applied: {DATE_FROM} — {DATE_TO}. "
            f"Rows before: {before_count}; rows after: {after_count}; "
            f"removed: {before_count - after_count}."
        )
    else:
        df["_published_at_dt_excel"] = ""
        FILTER_STATS["rows_before_date_filter"] = len(df)
        FILTER_STATS["rows_after_date_filter"] = len(df)
        FILTER_STATS["rows_removed_by_date_filter"] = 0

    df = detect_shorts(df)

    if EXCLUDE_SHORTS_ENABLED:
        before_short_count = len(df)
        shorts_count = int(df["_is_short_for_filter"].sum())

        df = df[~df["_is_short_for_filter"]].copy()

        after_short_count = len(df)

        FILTER_STATS["exclude_shorts_enabled"] = True
        FILTER_STATS["rows_before_short_filter"] = before_short_count
        FILTER_STATS["rows_removed_by_short_filter"] = shorts_count
        FILTER_STATS["rows_after_short_filter"] = after_short_count

        print(
            f"Shorts filter applied. "
            f"Rows before: {before_short_count}; rows after: {after_short_count}; "
            f"removed shorts/short-like: {shorts_count}."
        )
    else:
        FILTER_STATS["exclude_shorts_enabled"] = False
        FILTER_STATS["rows_before_short_filter"] = len(df)
        FILTER_STATS["rows_removed_by_short_filter"] = 0
        FILTER_STATS["rows_after_short_filter"] = len(df)

    if LIMIT_ROWS is not None:
        df = df.head(LIMIT_ROWS).copy()

    FILTER_STATS["rows_loaded_final"] = len(df)

    return df


def load_channel_ai() -> pd.DataFrame:
    if not CHANNEL_AI_XLSX.exists():
        print(f"WARNING: Channel AI file not found: {CHANNEL_AI_XLSX}")
        return pd.DataFrame()

    try:
        df = pd.read_excel(CHANNEL_AI_XLSX, sheet_name="channel_ai_analysis")
    except Exception as exc:
        print(f"WARNING: Could not read channel AI file: {exc}")
        return pd.DataFrame()

    if "channel_id" not in df.columns:
        print("WARNING: channel_ai_analysis has no channel_id column.")
        return pd.DataFrame()

    df["channel_id"] = df["channel_id"].astype(str).str.strip()

    keep_cols = [
        "channel_id",
        "channel_topic",
        "channel_function",
        "channel_type",
        "overall_relevance_score",
        "classification_confidence",
        "inclusion_status",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols].drop_duplicates(subset=["channel_id"], keep="first")


def load_playlist_ai() -> pd.DataFrame:
    if not PLAYLIST_AI_XLSX.exists():
        print(f"WARNING: Playlist AI file not found: {PLAYLIST_AI_XLSX}")
        return pd.DataFrame()

    try:
        df = pd.read_excel(PLAYLIST_AI_XLSX, sheet_name="playlist_ai_analysis")
    except Exception as exc:
        print(f"WARNING: Could not read playlist AI file: {exc}")
        return pd.DataFrame()

    if "playlist_id" not in df.columns:
        print("WARNING: playlist_ai_analysis has no playlist_id column.")
        return pd.DataFrame()

    df["playlist_id"] = df["playlist_id"].astype(str).str.strip()

    keep_cols = [
        "playlist_id",
        "playlist_topic",
        "playlist_function",
        "playlist_type",
        "playlist_relevance_score",
        "classification_confidence",
        "inclusion_status",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols].drop_duplicates(subset=["playlist_id"], keep="first")


def load_video_playlist_membership() -> pd.DataFrame:
    if not INPUT_XLSX.exists():
        raise FileNotFoundError(f"Input Excel file not found: {INPUT_XLSX}")

    xls = pd.ExcelFile(INPUT_XLSX)

    if SHEET_PLAYLIST_MEMBERSHIP not in xls.sheet_names:
        print(f"WARNING: Sheet not found: {SHEET_PLAYLIST_MEMBERSHIP}")
        return pd.DataFrame()

    df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_PLAYLIST_MEMBERSHIP)

    required = ["video_id", "playlist_id", "playlist_title"]
    validate_columns(df, required, SHEET_PLAYLIST_MEMBERSHIP)

    df["video_id"] = df["video_id"].astype(str).str.strip()
    df["playlist_id"] = df["playlist_id"].astype(str).str.strip()

    return df[["video_id", "playlist_id", "playlist_title"]].drop_duplicates()


# =========================
# KEYWORD FILTER
# =========================

def normalize_text_for_search(text: Any) -> str:
    text = safe_str(text).lower()
    text = text.replace("ґ", "г")
    text = re.sub(r"\s+", " ", text)
    return text


def find_keyword_hits(row: pd.Series) -> Tuple[int, str, str, int, str, int, str]:
    fields = [
        row.get("video_title", ""),
        row.get("description", ""),
        row.get("tags", ""),
        row.get("topic_categories", ""),
        row.get("channel_title", ""),
    ]

    text = normalize_text_for_search(" ".join(safe_str(x) for x in fields))

    all_hit_groups = []
    all_hit_terms = []

    direct_hit_terms = []
    direct_hit_groups = []

    context_hit_terms = []
    context_hit_groups = []

    for group_name, terms in KEYWORD_GROUPS.items():
        group_hits = []

        for term in terms:
            term_norm = normalize_text_for_search(term)

            if term_norm and term_norm in text:
                group_hits.append(term)

        if group_hits:
            all_hit_groups.append(group_name)
            all_hit_terms.extend(group_hits[:10])

            if group_name in DIRECT_KEYWORD_GROUPS:
                direct_hit_groups.append(group_name)
                direct_hit_terms.extend(group_hits[:10])

            if group_name in CONTEXT_ONLY_KEYWORD_GROUPS:
                context_hit_groups.append(group_name)
                context_hit_terms.extend(group_hits[:10])

    return (
        len(all_hit_terms),
        " | ".join(sorted(set(all_hit_groups))),
        " | ".join(sorted(set(all_hit_terms))),
        len(direct_hit_terms),
        " | ".join(sorted(set(direct_hit_groups))),
        len(context_hit_terms),
        " | ".join(sorted(set(context_hit_groups))),
    )


# =========================
# MERGE CONTEXT AND SELECT VIDEOS
# =========================

def prepare_selection_data() -> pd.DataFrame:
    uploads = load_uploads()

    channel_ai = load_channel_ai()

    if not channel_ai.empty:
        channel_ai = channel_ai.rename(columns={
            "overall_relevance_score": "channel_relevance_score",
            "classification_confidence": "channel_classification_confidence",
            "inclusion_status": "channel_inclusion_status",
        })

        uploads = uploads.merge(
            channel_ai,
            on="channel_id",
            how="left"
        )

    membership = load_video_playlist_membership()
    playlist_ai = load_playlist_ai()

    if not membership.empty and not playlist_ai.empty:
        membership_ai = membership.merge(
            playlist_ai,
            on="playlist_id",
            how="left"
        )

        grouped_rows = []

        for video_id, group in membership_ai.groupby("video_id", dropna=False):
            group = group.copy()

            score_series = pd.to_numeric(
                group.get("playlist_relevance_score", pd.Series(dtype=float)),
                errors="coerce"
            ).fillna(0)

            max_score = float(score_series.max()) if not score_series.empty else 0.0

            playlist_title_col = None
            for candidate in ["playlist_title", "playlist_title_x", "playlist_title_y"]:
                if candidate in group.columns:
                    playlist_title_col = candidate
                    break

            if playlist_title_col:
                playlist_titles = " | ".join(
                    sorted(set(group[playlist_title_col].dropna().astype(str).tolist()))
                )
            else:
                playlist_titles = ""

            grouped_rows.append({
                "video_id": safe_str(video_id),
                "playlist_max_relevance_score": max_score,
                "playlist_titles": playlist_titles,
                "playlist_count": int(len(group)),
            })

        video_playlist_context = pd.DataFrame(grouped_rows)

        uploads = uploads.merge(
            video_playlist_context,
            on="video_id",
            how="left"
        )

    numeric_context_cols = [
        "channel_relevance_score",
        "channel_classification_confidence",
        "playlist_max_relevance_score",
        "playlist_count",
    ]

    for col in numeric_context_cols:
        if col not in uploads.columns:
            uploads[col] = 0

        uploads[col] = pd.to_numeric(
            uploads[col],
            errors="coerce"
        ).fillna(0)

    text_context_cols = [
        "channel_topic",
        "channel_function",
        "channel_type",
        "channel_inclusion_status",
        "playlist_titles",
    ]

    for col in text_context_cols:
        if col not in uploads.columns:
            uploads[col] = ""

        uploads[col] = uploads[col].fillna("")

    hits = uploads.apply(
        find_keyword_hits,
        axis=1,
        result_type="expand"
    )

    uploads["keyword_hit_count"] = hits[0]
    uploads["keyword_hit_groups"] = hits[1]
    uploads["keyword_hit_terms"] = hits[2]

    uploads["direct_keyword_hit_count"] = hits[3]
    uploads["direct_keyword_hit_groups"] = hits[4]

    uploads["context_keyword_hit_count"] = hits[5]
    uploads["context_keyword_hit_groups"] = hits[6]

    direct_group_sets = uploads["direct_keyword_hit_groups"].apply(split_groups)

    uploads["strong_direct_group_count"] = direct_group_sets.apply(
        lambda groups: len(groups & STRONG_DIRECT_KEYWORD_GROUPS)
    )

    uploads["weak_direct_group_count"] = direct_group_sets.apply(
        lambda groups: len(groups & WEAK_DIRECT_KEYWORD_GROUPS)
    )

    uploads["selected_by_strong_direct_keywords"] = (
        uploads["strong_direct_group_count"] >= 1
    )

    uploads["selected_by_weak_direct_keywords"] = (
        (uploads["strong_direct_group_count"] == 0)
        & (uploads["weak_direct_group_count"] >= 2)
    )

    uploads["selected_by_direct_keywords"] = (
        uploads["selected_by_strong_direct_keywords"]
        | uploads["selected_by_weak_direct_keywords"]
    )

    uploads["selected_by_context_keywords_only"] = (
        (~uploads["selected_by_direct_keywords"])
        & (uploads["context_keyword_hit_count"] >= MIN_CONTEXT_ONLY_HITS)
    )

    uploads["selected_by_keywords"] = uploads["selected_by_direct_keywords"]

    uploads["selected_by_channel_context"] = (
        uploads["channel_relevance_score"] >= CHANNEL_RELEVANCE_THRESHOLD
    )

    uploads["selected_by_playlist_context"] = (
        uploads["playlist_max_relevance_score"] >= PLAYLIST_RELEVANCE_THRESHOLD
    )

    uploads["selected_by_context_supported_keywords"] = (
        (
            uploads["selected_by_context_keywords_only"]
            | (uploads["weak_direct_group_count"] >= 1)
        )
        & (
            uploads["selected_by_channel_context"]
            | uploads["selected_by_playlist_context"]
        )
    )

    uploads["selected_by_random_control"] = False

    base_selected = (
        uploads["selected_by_keywords"]
        | uploads["selected_by_context_supported_keywords"]
    )

    remaining = uploads[~base_selected].copy()

    if RANDOM_CONTROL_SAMPLE_SIZE > 0 and not remaining.empty:
        sample_n = min(RANDOM_CONTROL_SAMPLE_SIZE, len(remaining))

        sampled_idx = remaining.sample(
            n=sample_n,
            random_state=RANDOM_SEED
        ).index

        uploads.loc[sampled_idx, "selected_by_random_control"] = True

    uploads["send_to_ai"] = (
        uploads["selected_by_keywords"]
        | uploads["selected_by_context_supported_keywords"]
        | uploads["selected_by_random_control"]
    )

    selection_reasons = []

    for _, row in uploads.iterrows():
        reasons = []

        if bool(row.get("selected_by_strong_direct_keywords", False)):
            reasons.append("strong_direct_keywords")

        if bool(row.get("selected_by_weak_direct_keywords", False)):
            reasons.append("weak_direct_keywords_combined")

        if bool(row.get("selected_by_context_keywords_only", False)):
            reasons.append("context_keywords_only_not_selected")

        if bool(row.get("selected_by_context_supported_keywords", False)):
            reasons.append("context_supported_keywords")

        if bool(row.get("selected_by_random_control", False)):
            reasons.append("random_control")

        selection_reasons.append(" | ".join(reasons))

    uploads["selection_reason"] = selection_reasons

    return uploads


# =========================
# PROMPT BUILDING
# =========================

def make_row_key(row: pd.Series) -> str:
    video_id = safe_str(row.get("video_id", ""))

    if video_id:
        return video_id

    return f"{safe_str(row.get('channel_id', ''))}::{safe_str(row.get('video_title', ''))}"


def infer_format(row: pd.Series) -> str:
    if bool(row.get("_is_short_for_filter", False)):
        return "short_video"

    source_section = safe_str(row.get("source_section", "")).lower()
    api_section = safe_str(row.get("api_section", "")).lower()
    is_stream = safe_str(row.get("is_stream", "")).lower()
    duration = safe_float(row.get("duration_seconds", 0), 0)

    if "stream" in source_section or "stream" in api_section or is_stream in {"true", "1", "yes"}:
        return "stream_or_live_recording"

    if duration and duration <= SHORT_MAX_DURATION_SECONDS:
        return "possible_short_or_clip"

    return "long_video_or_standard_upload"


def build_user_prompt(row: pd.Series) -> str:
    data = {
        "selection_reason": safe_str(row.get("selection_reason", "")),
        "keyword_hit_groups": safe_str(row.get("keyword_hit_groups", "")),
        "keyword_hit_terms": safe_str(row.get("keyword_hit_terms", "")),
        "direct_keyword_hit_groups": safe_str(row.get("direct_keyword_hit_groups", "")),
        "context_keyword_hit_groups": safe_str(row.get("context_keyword_hit_groups", "")),

        "video_id": safe_str(row.get("video_id", "")),
        "video_url": safe_str(row.get("video_url", "")),
        "channel_id": safe_str(row.get("channel_id", "")),
        "channel_title": safe_str(row.get("channel_title", "")),
        "video_title": safe_str(row.get("video_title", "")),
        "video_description": truncate_text(row.get("description", ""), 3000),
        "tags": truncate_text(row.get("tags", ""), 1500),
        "topic_categories": truncate_text(row.get("topic_categories", ""), 1000),

        "published_at": safe_str(row.get("published_at", "")),
        "duration_seconds": safe_str(row.get("duration_seconds", "")),
        "inferred_video_format": infer_format(row),
        "source_section": safe_str(row.get("source_section", "")),
        "api_section": safe_str(row.get("api_section", "")),

        "view_count": safe_str(row.get("view_count", "")),
        "like_count": safe_str(row.get("like_count", "")),
        "comment_count": safe_str(row.get("comment_count", "")),

        "channel_ai_context": {
            "channel_topic": safe_str(row.get("channel_topic", "")),
            "channel_function": safe_str(row.get("channel_function", "")),
            "channel_type": safe_str(row.get("channel_type", "")),
            "channel_relevance_score": safe_str(row.get("channel_relevance_score", "")),
            "channel_inclusion_status": safe_str(row.get("channel_inclusion_status", "")),
        },

        "playlist_ai_context": {
            "playlist_max_relevance_score": safe_str(row.get("playlist_max_relevance_score", "")),
            "playlist_titles": truncate_text(row.get("playlist_titles", ""), 1500),
            "playlist_count": safe_str(row.get("playlist_count", "")),
        },
    }

    return (
        "Classify the following YouTube video metadata for the research topic:\n"
        "\"Social practices of public audit and corruption exposure in Ukraine.\"\n\n"
        "VIDEO METADATA JSON:\n"
        f"{json.dumps(data, ensure_ascii=False, indent=2)}"
    )


# =========================
# OPENAI CALL
# =========================

def parse_json_response(text: str) -> Dict[str, Any]:
    text = text.strip()

    if text.startswith("```"):
        text = text.strip("`").strip()

        if text.lower().startswith("json"):
            text = text[4:].strip()

    return json.loads(text)


def call_model(client: OpenAI, row: pd.Series) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": build_user_prompt(row)},
        ],
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        timeout=REQUEST_TIMEOUT,
    )

    content = response.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Model returned empty response.")

    return parse_json_response(content)


# =========================
# OUTPUT HELPERS
# =========================

def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_XLSX.exists():
        return pd.DataFrame()

    try:
        return pd.read_excel(OUTPUT_XLSX, sheet_name="video_ai_analysis")
    except Exception:
        return pd.DataFrame()


def existing_keys(existing_df: pd.DataFrame) -> set:
    if existing_df.empty or "row_key" not in existing_df.columns:
        return set()

    return set(existing_df["row_key"].dropna().astype(str).tolist())


def save_outputs(results: List[Dict[str, Any]], log_rows: List[Dict[str, Any]]) -> None:
    results_df = pd.DataFrame(results)
    log_df = pd.DataFrame(log_rows)

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        results_df.to_excel(writer, index=False, sheet_name="video_ai_analysis")

    log_df.to_excel(LOG_XLSX, index=False)


def save_selection_diagnostics(df: pd.DataFrame) -> None:
    summary_rows = [
        {"metric": "rows_initial", "value": FILTER_STATS.get("rows_initial", "")},
        {"metric": "date_filter_enabled", "value": DATE_FILTER_ENABLED},
        {"metric": "date_from", "value": DATE_FROM},
        {"metric": "date_to", "value": DATE_TO},
        {"metric": "rows_after_date_filter", "value": FILTER_STATS.get("rows_after_date_filter", "")},
        {"metric": "exclude_shorts_enabled", "value": EXCLUDE_SHORTS_ENABLED},
        {"metric": "short_max_duration_seconds", "value": SHORT_MAX_DURATION_SECONDS},
        {"metric": "rows_removed_by_short_filter", "value": FILTER_STATS.get("rows_removed_by_short_filter", "")},
        {"metric": "uploads_rows_total_after_filters", "value": len(df)},

        {"metric": "selected_by_keywords", "value": int(df["selected_by_keywords"].sum())},
        {"metric": "selected_by_strong_direct_keywords", "value": int(df["selected_by_strong_direct_keywords"].sum())},
        {"metric": "selected_by_weak_direct_keywords", "value": int(df["selected_by_weak_direct_keywords"].sum())},
        {"metric": "selected_by_direct_keywords", "value": int(df["selected_by_direct_keywords"].sum())},
        {"metric": "context_keywords_only_not_selected", "value": int(df["selected_by_context_keywords_only"].sum())},
        {"metric": "selected_by_context_supported_keywords", "value": int(df["selected_by_context_supported_keywords"].sum())},
        {"metric": "selected_by_channel_context", "value": int(df["selected_by_channel_context"].sum())},
        {"metric": "selected_by_playlist_context", "value": int(df["selected_by_playlist_context"].sum())},
        {"metric": "selected_by_random_control", "value": int(df["selected_by_random_control"].sum())},
        {"metric": "send_to_ai_total", "value": int(df["send_to_ai"].sum())},
    ]

    summary = pd.DataFrame(summary_rows)

    selected_preview_cols = [
        "video_id",
        "channel_title",
        "video_title",
        "published_at",
        "_published_at_dt_excel",
        "_is_short_for_filter",
        "_short_filter_reason",
        "duration_seconds",
        "source_section",
        "api_section",
        "selection_reason",

        "keyword_hit_count",
        "keyword_hit_groups",
        "keyword_hit_terms",

        "direct_keyword_hit_count",
        "direct_keyword_hit_groups",
        "context_keyword_hit_count",
        "context_keyword_hit_groups",

        "strong_direct_group_count",
        "weak_direct_group_count",
        "selected_by_strong_direct_keywords",
        "selected_by_weak_direct_keywords",
        "selected_by_direct_keywords",
        "selected_by_context_keywords_only",
        "selected_by_context_supported_keywords",

        "channel_relevance_score",
        "playlist_max_relevance_score",

        "view_count",
        "like_count",
        "comment_count",
    ]

    selected_preview_cols = [c for c in selected_preview_cols if c in df.columns]

    with pd.ExcelWriter(DIAGNOSTICS_XLSX, engine="openpyxl") as writer:
        summary.to_excel(writer, index=False, sheet_name="summary")

        df[df["send_to_ai"]].head(1000)[selected_preview_cols].to_excel(
            writer,
            index=False,
            sheet_name="selected_preview"
        )

        df[~df["send_to_ai"]].head(1000)[selected_preview_cols].to_excel(
            writer,
            index=False,
            sheet_name="not_selected_preview"
        )

        df[df["selected_by_strong_direct_keywords"]].head(1000)[selected_preview_cols].to_excel(
            writer,
            index=False,
            sheet_name="strong_direct_preview"
        )

        df[df["selected_by_context_supported_keywords"]].head(1000)[selected_preview_cols].to_excel(
            writer,
            index=False,
            sheet_name="context_supported_preview"
        )

    print(f"Selection diagnostics saved to: {DIAGNOSTICS_XLSX}")


# =========================
# MAIN
# =========================

def main() -> None:
    df = prepare_selection_data()
    save_selection_diagnostics(df)

    selected_total = int(df["send_to_ai"].sum())
    total_rows = len(df)
    selected_share = selected_total / total_rows if total_rows else 0

    print("\n=== SELECTION SUMMARY ===")
    print(f"Date filter enabled: {DATE_FILTER_ENABLED}")
    print(f"Date range: {DATE_FROM} — {DATE_TO}")
    print(f"Exclude shorts enabled: {EXCLUDE_SHORTS_ENABLED}")
    print(f"Short max duration seconds: {SHORT_MAX_DURATION_SECONDS}")
    print(f"Rows initial: {FILTER_STATS.get('rows_initial', '')}")
    print(f"Rows after date filter: {FILTER_STATS.get('rows_after_date_filter', '')}")
    print(f"Rows removed by short filter: {FILTER_STATS.get('rows_removed_by_short_filter', '')}")
    print(f"Uploads loaded after filters: {total_rows}")
    print(f"Selected for AI: {selected_total}")
    print(f"Selected share: {selected_share:.2%}")

    print(f"Selected by keywords: {int(df['selected_by_keywords'].sum())}")
    print(f"Selected by strong direct keywords: {int(df['selected_by_strong_direct_keywords'].sum())}")
    print(f"Selected by weak direct keywords: {int(df['selected_by_weak_direct_keywords'].sum())}")
    print(f"Selected by context-supported keywords: {int(df['selected_by_context_supported_keywords'].sum())}")
    print(f"Selected by channel context: {int(df['selected_by_channel_context'].sum())}")
    print(f"Selected by playlist context: {int(df['selected_by_playlist_context'].sum())}")
    print(f"Selected by random control: {int(df['selected_by_random_control'].sum())}")

    print(f"Diagnostics saved to: {DIAGNOSTICS_XLSX}")

    if DRY_RUN_ONLY:
        print("\nDRY_RUN_ONLY = True, so no OpenAI API calls were made.")
        return

    api_key = load_api_key()
    client = OpenAI(api_key=api_key)

    ai_df = df[df["send_to_ai"]].copy()

    if MAX_AI_ROWS is not None:
        ai_df = ai_df.head(MAX_AI_ROWS).copy()

    existing_df = load_existing_results()
    processed_keys = existing_keys(existing_df) if SKIP_EXISTING else set()

    results: List[Dict[str, Any]] = []
    log_rows: List[Dict[str, Any]] = []

    if not existing_df.empty and SKIP_EXISTING:
        results.extend(existing_df.to_dict(orient="records"))

    print(f"\nUploads loaded after filters: {len(df)}")
    print(f"Selected for AI: {int(df['send_to_ai'].sum())}")
    print(f"Will process in this run: {len(ai_df)}")
    print(f"Already processed: {len(processed_keys)}")
    print(f"Output: {OUTPUT_XLSX}")

    for processed_n, (idx, row) in enumerate(ai_df.iterrows(), start=1):
        row_key = make_row_key(row)

        if not row_key:
            log_rows.append({
                "row_index": idx,
                "row_key": "",
                "video_id": safe_str(row.get("video_id", "")),
                "video_title": safe_str(row.get("video_title", "")),
                "status": "failed",
                "error": "Cannot construct row_key",
            })
            continue

        if SKIP_EXISTING and row_key in processed_keys:
            print(f"Skipping existing: {row_key}")
            continue

        print(
            f"Processing {processed_n}/{len(ai_df)} "
            f"(source row {idx}): {safe_str(row.get('video_title', ''))[:100]}"
        )

        last_error: Optional[Exception] = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                ai = call_model(client, row)

                result_row = {
                    "row_key": row_key,
                    "source_row_index": idx,

                    "video_id": safe_str(row.get("video_id", "")),
                    "video_url": safe_str(row.get("video_url", "")),
                    "channel_id": safe_str(row.get("channel_id", "")),
                    "channel_title": safe_str(row.get("channel_title", "")),
                    "video_title": safe_str(row.get("video_title", "")),
                    "description": truncate_text(row.get("description", ""), 2000),
                    "published_at": safe_str(row.get("published_at", "")),
                    "duration_seconds": safe_str(row.get("duration_seconds", "")),
                    "source_section": safe_str(row.get("source_section", "")),
                    "api_section": safe_str(row.get("api_section", "")),
                    "view_count": safe_str(row.get("view_count", "")),
                    "like_count": safe_str(row.get("like_count", "")),
                    "comment_count": safe_str(row.get("comment_count", "")),
                    "tags": truncate_text(row.get("tags", ""), 1000),
                    "topic_categories": truncate_text(row.get("topic_categories", ""), 1000),

                    "selection_reason": safe_str(row.get("selection_reason", "")),
                    "keyword_hit_count": safe_str(row.get("keyword_hit_count", "")),
                    "keyword_hit_groups": safe_str(row.get("keyword_hit_groups", "")),
                    "keyword_hit_terms": safe_str(row.get("keyword_hit_terms", "")),
                    "direct_keyword_hit_count": safe_str(row.get("direct_keyword_hit_count", "")),
                    "direct_keyword_hit_groups": safe_str(row.get("direct_keyword_hit_groups", "")),
                    "context_keyword_hit_count": safe_str(row.get("context_keyword_hit_count", "")),
                    "context_keyword_hit_groups": safe_str(row.get("context_keyword_hit_groups", "")),

                    "strong_direct_group_count": safe_str(row.get("strong_direct_group_count", "")),
                    "weak_direct_group_count": safe_str(row.get("weak_direct_group_count", "")),

                    "channel_relevance_score": safe_str(row.get("channel_relevance_score", "")),
                    "playlist_max_relevance_score": safe_str(row.get("playlist_max_relevance_score", "")),
                    "playlist_titles": truncate_text(row.get("playlist_titles", ""), 1000),

                    "video_topic": ai.get("video_topic", ""),
                    "secondary_topics": list_to_pipe(ai.get("secondary_topics", [])),
                    "video_function": ai.get("video_function", ""),
                    "video_format_type": ai.get("video_format_type", ""),
                    "corruption_relevance_category": ai.get("corruption_relevance_category", ""),
                    "public_audit_relevance_category": ai.get("public_audit_relevance_category", ""),
                    "ukraine_relevance_category": ai.get("ukraine_relevance_category", ""),
                    "evidence_type": ai.get("evidence_type", ""),
                    "video_relevance_score": ai.get("video_relevance_score", ""),
                    "classification_confidence": ai.get("classification_confidence", ""),
                    "inclusion_status": ai.get("inclusion_status", ""),
                    "manual_review_required": ai.get("manual_review_required", ""),
                    "evidence_markers": list_to_pipe(ai.get("evidence_markers", [])),
                    "risk_notes": list_to_pipe(ai.get("risk_notes", [])),
                    "short_rationale": ai.get("short_rationale", ""),
                }

                results.append(result_row)

                log_rows.append({
                    "row_index": idx,
                    "row_key": row_key,
                    "video_id": safe_str(row.get("video_id", "")),
                    "video_title": safe_str(row.get("video_title", "")),
                    "status": "ok",
                    "error": "",
                })

                last_error = None
                break

            except Exception as exc:
                last_error = exc
                wait_seconds = attempt * 3

                print(f"  Attempt {attempt} failed: {exc}")

                if attempt < MAX_RETRIES:
                    print(f"  Retrying in {wait_seconds} seconds...")
                    time.sleep(wait_seconds)

        if last_error is not None:
            log_rows.append({
                "row_index": idx,
                "row_key": row_key,
                "video_id": safe_str(row.get("video_id", "")),
                "video_title": safe_str(row.get("video_title", "")),
                "status": "failed",
                "error": str(last_error),
            })

        save_outputs(results, log_rows)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print("Done.")
    print(f"Results saved to: {OUTPUT_XLSX}")
    print(f"Log saved to: {LOG_XLSX}")
    print(f"Diagnostics saved to: {DIAGNOSTICS_XLSX}")


if __name__ == "__main__" and RUN_ANALYSIS:
    prepare_run(MODEL_NAME, OUTPUT_XLSX, LOG_XLSX)
    main()
elif __name__ == "__main__":
    print("Preview mode: review configuration, then set RUN_ANALYSIS = True to execute this stage.")